In [18]:
import torch
import torch.nn as nn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ──────────────────────────────────────────────
# 1. Multi-Layer Neural Network from scratch
# ──────────────────────────────────────────────
class MultiLayerNet(nn.Module):
    """
    A flexible multi-layer neural network for binary classification.

    Forward propagation flow:
        Input → [Linear → ReLU → Dropout] × N hidden layers → Linear → Sigmoid → Output
    """

    def __init__(self, layer_sizes: list[int], dropout: float = 0.0):
        """
        Args:
            layer_sizes: list of ints, e.g. [4, 16, 8, 1]
                         first = input features, last = output (1 for binary)
            dropout:     dropout probability between hidden layers
        """
        super().__init__()

        layers = []
        for i in range(len(layer_sizes) - 1):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i + 1]))

            if i < len(layer_sizes) - 2:          # hidden layers only
                layers.append(nn.ReLU())
                if dropout > 0:
                    layers.append(nn.Dropout(dropout))

        layers.append(nn.Sigmoid())                # final activation
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward propagation — data flows through every layer in sequence:
          x  →  Linear₁ → ReLU → Linear₂ → ReLU → … → Linearₙ → Sigmoid
        """
        return self.network(x)

    def predict(self, x: torch.Tensor) -> torch.Tensor:
        self.eval()
        with torch.no_grad():
            return (self.forward(x) >= 0.5).float()


# ──────────────────────────────────────────────
# 2. Step-by-step manual forward pass (educational)
# ──────────────────────────────────────────────
def manual_forward(model: MultiLayerNet, x: torch.Tensor) -> torch.Tensor:
    """
    Walk through forward propagation layer-by-layer,
    printing the shape and activation at each stage.
    """
    h = x
    print(f"{'Input':<18} shape: {list(h.shape)}")

    for i, layer in enumerate(model.network):
        h = layer(h)
        name = layer.__class__.__name__
        print(f"  → {name:<14} shape: {list(h.shape)}   "
              f"mean={h.mean().item():+.4f}  std={h.std().item():.4f}")

    return h


# ──────────────────────────────────────────────
# 3. Example usage
# ──────────────────────────────────────────────
if __name__ == "__main__":

    # --- Toy dataset: 6 features, 1000 samples ---
    X, y = make_classification(
        n_samples=1000, n_features=6, n_informative=4,
        n_redundant=1, n_clusters_per_class=2, random_state=42,
    )

    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42,
    )

    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_test  = torch.tensor(X_test,  dtype=torch.float32)
    y_test  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)

    # --- Build a 3-hidden-layer network: 6 → 32 → 16 → 8 → 1 ---
    model = MultiLayerNet(
        layer_sizes=[6, 32, 16, 8, 1],
        dropout=0.1,
    )
    print(model)
    print(f"Total params: {sum(p.numel() for p in model.parameters()):,}\n")

    # --- Visualise one forward pass ---
    print("── Manual forward pass on a single sample ──")
    sample = X_train[:1]
    _ = manual_forward(model, sample)
    print()

    # --- Train ---
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    EPOCHS = 80
    for epoch in range(1, EPOCHS + 1):
        model.train()

        y_pred = model(X_train)              # ← forward propagation
        loss   = criterion(y_pred, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            acc = (model.predict(X_train) == y_train).float().mean()
            print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f} | Train Acc: {acc:.2%}")

    # --- Evaluate ---
    test_acc = (model.predict(X_test) == y_test).float().mean()
    print(f"\nTest Accuracy: {test_acc:.2%}")

MultiLayerNet(
  (network): Sequential(
    (0): Linear(in_features=6, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=16, out_features=8, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.1, inplace=False)
    (9): Linear(in_features=8, out_features=1, bias=True)
    (10): Sigmoid()
  )
)
Total params: 897

── Manual forward pass on a single sample ──
Input              shape: [1, 6]
  → Linear         shape: [1, 32]   mean=-0.0717  std=0.9682
  → ReLU           shape: [1, 32]   mean=+0.3562  std=0.5784
  → Dropout        shape: [1, 32]   mean=+0.3958  std=0.6427
  → Linear         shape: [1, 16]   mean=-0.0841  std=0.4163
  → ReLU           shape: [1, 16]   mean=+0.1451  std=0.2259
  → Dropout        shape: [1, 16]   mean=+0.1545  std=0.2540
  → Linear         shape: [1, 8]   mean=-0.0501  std=0.2369
  → ReLU  

C:\Users\nikun\AppData\Local\Temp\ipykernel_8688\2195738047.py:67: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1858.)
  f"mean={h.mean().item():+.4f}  std={h.std().item():.4f}")


Epoch  80 | Loss: 0.1811 | Train Acc: 95.00%

Test Accuracy: 91.00%
